In [ ]:
from sklearn.ensemble        import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import SVC
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics         import (accuracy_score,
                                    f1_score,
                                    matthews_corrcoef,
                                    precision_score,recall_score,
                                    confusion_matrix)
import joblib
import pandas as pd
import numpy as np

np.random.seed(42)

In [ ]:
root = '.'
df = pd.read_csv(root)
X_train = df.to_numpy()

y_train = pd.read_csv(root + "labels")
y_train

df = pd.read_csv(root)
X_valid = df.to_numpy()

y_valid = pd.read_csv(root + "labels")
y_valid 

In [ ]:
(X_train.shape,X_valid.shape,y_train.shape,y_valid.shape)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid  = scaler.transform(X_valid)


use_pca = True
if use_pca:
    pca = PCA(0.95)
    
    X_train = pca.fit_transform(X_train)
    X_valid = pca.transform(X_valid)

In [ ]:
classifiers = [RandomForestClassifier(random_state=42),
               LogisticRegression(random_state=42),
               SVC(random_state=42), 
               GradientBoostingClassifier(random_state=42)]

params = [
    {
        
    },
    {
        
    },
    {
        
    },
    {
        
    }
]

cv = StratifiedKFold(n=5)

classifiers_stats = {}

for classifier, param in zip(classifiers, params):
    
    grid = GridSearchCV(classifier, param_grid=param, scoring=matthews_corrcoef, cv = cv, refit=True)
    
    grid.fit(X_train,y_train)
    y_pred = grid.predict(X_valid)
    
    
    acc = accuracy_score(y_valid,y_pred)
    f1s = f1_score(y_valid,y_pred)
    mcc = matthews_corrcoef(y_valid,y_pred)
    prc = precision_score(y_valid,y_pred)
    rec = recall_score(y_valid,y_pred)
    cmf = confusion_matrix(y_valid,y_pred)
    
    print(f'classifier: {type(grid.best_estimator_).__name__} performance: ')
    print(acc)
    print(f1s)
    print(mcc)
    print(prc)
    print(rec)
    print(cmf)
    
    classifiers_stats[f'{type(grid.best_estimator_).__name__}'] = {'model' : grid.best_estimator_,
                                                                   'acc' : acc,
                                                                   'f1s' : f1s,
                                                                   'mcc' : mcc,
                                                                   'prc' : prc,
                                                                   'rec' : rec,
                                                                   'cmf' : cmf,}
    
    
filename = 'classification_results_PCA' if use_pca else 'classification_results'
joblib.dump(classifiers_stats,filename)